## Imports

In [ ]:
import os
import torch
import torchvision
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import torch.optim as optim
import shutil
from torchvision import transforms
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

## Test GPU

In [ ]:
# === Vérification GPU ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    print(f'✅ GPU détecté : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM dispo  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   CUDA version : {torch.version.cuda}')
else:
    print('⚠️  Aucun GPU détecté — entraînement sur CPU (lent)')

print(f'\n👉 Device utilisé : {device}')

## Téléchargement des données

In [ ]:
# === Téléchargement ===
path = kagglehub.dataset_download("phucthaiv02/butterfly-image-classification")
path = Path(path)
print(f"✅ Dataset téléchargé : {path}")

# === On supprime test car on n'a pas les labels donc on split sur les train ===
test_dir = path / "test"
if test_dir.exists():
    shutil.rmtree(test_dir)
    print(f"🗑️  Dossier test supprimé")

## Transformation des données et split

In [ ]:
# === Transforms ===
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# === Dataset ===
class ButterflyDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.classes = sorted(self.df["label"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir / row["filename"]
        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[row["label"]]
        if self.transform:
            image = self.transform(image)
        return image, label

# === Deux instances séparées (bonne pratique) ===
csv_file = path / "Training_set.csv"
img_dir  = path / "train"

train_val_dataset = ButterflyDataset(csv_file, img_dir, transform=train_transform)
eval_dataset      = ButterflyDataset(csv_file, img_dir, transform=eval_transform)

NUM_CLASSES  = len(train_val_dataset.classes)
CLASS_NAMES  = train_val_dataset.classes
n            = len(train_val_dataset)

# === Split 70 / 15 / 15 ===
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

indices = torch.randperm(n, generator=torch.Generator().manual_seed(42)).tolist()

from torch.utils.data import Subset
train_set = Subset(train_val_dataset, indices[:n_train])
val_set   = Subset(eval_dataset,      indices[n_train:n_train + n_val])
test_set  = Subset(eval_dataset,      indices[n_train + n_val:])

# === DataLoaders ===
train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"\n✅ Classes     : {NUM_CLASSES}")
print(f"   Train       : {len(train_set)} images")
print(f"   Validation  : {len(val_set)} images")
print(f"   Test        : {len(test_set)} images")

## Fonction d'entraînement

In [ ]:
import torch.nn as nn
def train_model(model, train_loader, val_loader, num_epochs=25, lr=5e-4, model_name="model"):
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_weights = None

    for epoch in range(num_epochs):
        # === TRAIN ===
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        pbar = tqdm(train_loader, desc=f"[{model_name}] Epoch {epoch+1}/{num_epochs}")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item() * images.size(0)
            preds          = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total   += labels.size(0)

            pbar.set_postfix({
                "loss": f"{train_loss/train_total:.4f}",
                "acc":  f"{train_correct/train_total:.4f}"
            })

        train_loss /= train_total
        train_acc   = train_correct / train_total

        # === VALIDATION ===
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss    = criterion(outputs, labels)

                val_loss    += loss.item() * images.size(0)
                preds        = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)

        val_loss /= val_total
        val_acc   = val_correct / val_total

        scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"  Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    model.load_state_dict(best_weights)
    print(f"\n✅ Meilleure val accuracy : {best_val_acc:.4f}")
    return model, history

In [ ]:
def plot_history(history, model_name="model"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history["train_loss"], label="Train")
    ax1.plot(history["val_loss"],   label="Validation")
    ax1.set_title(f"{model_name} — Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history["train_acc"], label="Train")
    ax2.plot(history["val_acc"],   label="Validation")
    ax2.set_title(f"{model_name} — Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True)

    plt.suptitle(f"{model_name} — Courbes d'apprentissage", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{model_name}_curves.png", dpi=150)
    plt.show()

## Fonction d'évaluation

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report, confusion_matrix
import seaborn as sns

def evaluate_model(model, test_loader, class_names, model_name="model"):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Évaluation {model_name}"):
            images = images.to(device)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    kappa    = cohen_kappa_score(all_labels, all_preds)

    print(f"\n📊 Résultats — {model_name}")
    print(f"   Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Kappa    : {kappa:.4f}")
    print(f"\n📋 Classification Report :")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(20, 16))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{model_name} — Matrice de confusion")
    plt.ylabel("Vrai label")
    plt.xlabel("Prédiction")
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(rotation=0,  fontsize=6)
    plt.tight_layout()
    plt.savefig(f"{model_name}_confusion_matrix.png", dpi=150)
    plt.show()

    return {"accuracy": accuracy, "kappa": kappa, "preds": all_preds, "labels": all_labels}

# CNN 

In [ ]:
import torch.nn as nn

class ButterflyNet(nn.Module):
    def __init__(self, num_classes=75):
        super(ButterflyNet, self).__init__()

        self.features = nn.Sequential(
            # Bloc 1 — 224x224 -> 112x112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Bloc 2 — 112x112 -> 56x56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Bloc 3 — 56x56 -> 28x28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Bloc 4 — 28x28 -> 14x14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Bloc 5 — 14x14 -> 7x7
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  
            nn.Flatten(),             
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## Entraînement du CNN

In [ ]:
model_cnn = ButterflyNet(num_classes=NUM_CLASSES).to(device)
model_cnn, history_cnn = train_model(model_cnn, train_loader, val_loader, num_epochs=25, model_name="CNN")
plot_history(history_cnn, model_name="CNN")

In [ ]:
# Sauvegarder les poids du CNN
torch.save(model_cnn.state_dict(), "cnn_weights.pth")
print("   Poids sauvegardés : cnn_weights.pth")

## Evaluation du CNN

In [ ]:
results_cnn = evaluate_model(model_cnn, test_loader, CLASS_NAMES, model_name="CNN")

# EfficentNetB0

## Chargement du modèle

In [ ]:
from torchvision.models import EfficientNet_B0_Weights

model_eff = torchvision.models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# Geler les couches convolutives
for param in model_eff.features.parameters():
    param.requires_grad = False

# Remplacer la tête de classification
model_eff.classifier[1] = nn.Linear(in_features=1280, out_features=NUM_CLASSES)
model_eff = model_eff.to(device)

print(f"   EfficientNetB0 chargé pour {NUM_CLASSES} classes")
print(f"   Paramètres entraînables : {sum(p.numel() for p in model_eff.parameters() if p.requires_grad):,}")
print(f"   Paramètres gelés        : {sum(p.numel() for p in model_eff.parameters() if not p.requires_grad):,}")

## Train de EfficientNetB0

In [ ]:
model_eff, history_eff = train_model(
    model_eff, train_loader, val_loader,
    num_epochs=25, lr=1e-3, model_name="EfficientNetB0"
)
plot_history(history_eff, model_name="EfficientNetB0")

In [ ]:
torch.save(model_eff.state_dict(), "efficientnet_weights.pth")
print("   Poids sauvegardés")

## Evaluation de EfficientNetB0 

In [ ]:
results_eff = evaluate_model(model_eff, test_loader, CLASS_NAMES, model_name="EfficientNetB0")